# Merged Data + Domain Segmentation

Combines the two experiments from nb42 and nb43:
- **nb42**: Domain-specific models on AUB-only data → selective segmentation gave ~+0.77 F1 points
- **nb43**: Merged peer university data (Lehigh, Marquette, Villanova) → ~0 change on AUB F1

**Hypothesis**: Training domain-specific models on the larger merged dataset gives each domain model more samples, potentially enabling the domain-specific approach to outperform both the AUB-only domain models and the unified merged model.

**Reference baselines:**
| Model | Train | Test | F1 |
|---|---|---|---|
| AUB-only baseline | AUB 2015-2017 | AUB 2018-2020 | 62.55% |
| Merged (nb43) | All-unis 2015-2017 | AUB 2018-2020 | 62.53% |
| Domain segmentation (nb42) | AUB 2010-2017 | AUB 2018-2020 | 63.33% |

**Two scenarios tested:**
- **Scenario A** — 2015-2017 merged train (fair comparison to nb43)
- **Scenario B** — 2010-2017 merged train (maximum data, fair comparison to nb42)

In [ ]:
import sys
sys.path.append('../')

import pandas as pd
import numpy as np
import pickle
import warnings
from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    roc_auc_score, confusion_matrix, accuracy_score
)

import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8')
%matplotlib inline

# ── Reference baselines ────────────────────────────────────────────────────────
BASELINES = {
    'AUB-only baseline':           {'f1': 0.6255, 'roc_auc': 0.8104, 'recall': 0.7715, 'precision': 0.5258},
    'Merged model - AUB test (nb43)': {'f1': 0.6253, 'roc_auc': 0.8100, 'recall': 0.7700, 'precision': 0.5258},
    'Domain segmentation (nb42)':  {'f1': 0.6333, 'roc_auc': None,   'recall': None,   'precision': None},
}

## 1. Load Merged Data

In [ ]:
data_path = Path('../data/processed/all_unis_cleaned.pkl')

if not data_path.exists():
    raise FileNotFoundError(
        f"Merged data not found: {data_path}\n"
        "Run notebooks 04 → 05 → 06 first."
    )

df = pd.read_pickle(data_path)

print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} cols")
print(f"\nInstitutions:")
print(df['institution'].value_counts().to_string())
print(f"\nYear range: {df['Year'].min()} – {df['Year'].max()}")

## 2. Domain Mapping

Same substring-matching function as nb42.

In [ ]:
# Identify ASJC column
asjc_col = None
if 'All Science Journal Classification (ASJC) field name' in df.columns:
    asjc_col = 'All Science Journal Classification (ASJC) field name'
elif 'ASJC field name' in df.columns:
    asjc_col = 'ASJC field name'
else:
    candidates = [c for c in df.columns if 'asjc' in c.lower()]
    if candidates:
        asjc_col = candidates[0]

print(f"ASJC column: '{asjc_col}'")


def map_to_domain(asjc_field):
    """Map specific ASJC field values to broader research domains (same as nb42)."""
    if pd.isna(asjc_field):
        return 'Other'
    field_lower = str(asjc_field).lower()

    if 'multidisciplinary' in field_lower:
        return 'Multidisciplinary'

    medicine_terms = [
        'medicine', 'surgery', 'nursing', 'health', 'cardiology', 'cardiovascular',
        'oncology', 'cancer', 'radiology', 'nuclear medicine', 'anesthesiology',
        'obstetrics', 'gynecology', 'urology', 'ophthalmology', 'hematology',
        'epidemiology', 'emergency', 'gastroenterology', 'hepatology',
        'rheumatology', 'orthopedic', 'dermatology', 'psychiatry', 'neurology',
        'pediatrics', 'otorhinolaryngology', 'infectious diseases', 'pulmonary',
        'respiratory', 'critical care', 'intensive care', 'pharmacology',
        'immunology', 'allergy', 'transplantation', 'pathology', 'anatomy',
        'physiology', 'physical therapy', 'rehabilitation', 'dentistry',
        'endocrinology', 'nephrology', 'geriatrics', 'palliative',
        'clinical', 'medical', 'hospital', 'patient', 'diagnosis', 'treatment',
        'microbiology (medical)', 'genetics', 'general nursing'
    ]
    if any(t in field_lower for t in medicine_terms):
        return 'Medicine & Health'

    engineering_terms = [
        'engineering', 'electrical', 'electronic', 'mechanical', 'civil',
        'chemical engineering', 'aerospace', 'biomedical engineering',
        'industrial', 'manufacturing', 'control and systems', 'automation',
        'telecommunications', 'signal processing', 'computer science',
        'information systems', 'software', 'hardware', 'artificial intelligence',
        'machine learning', 'computational', 'materials science', 'energy',
        'renewable energy', 'nuclear energy', 'robotics', 'mechatronics'
    ]
    if any(t in field_lower for t in engineering_terms):
        return 'Engineering & Technology'

    social_terms = [
        'education', 'psychology', 'economics', 'business', 'management',
        'social science', 'communication', 'policy', 'political', 'sociology',
        'anthropology', 'history', 'philosophy', 'linguistics', 'law',
        'public administration', 'cultural', 'media', 'journalism',
        'library', 'information science', 'tourism', 'sport', 'geography',
        'demography', 'urban', 'development studies', 'gender', 'religion',
        'arts and humanities', 'architecture', 'urban planning',
        'accounting', 'finance', 'marketing', 'strategy', 'organizational',
        'human resource', 'supply chain', 'operations research',
        'health (social science)'
    ]
    if any(t in field_lower for t in social_terms):
        return 'Social Sciences'

    natural_terms = [
        'chemistry', 'physics', 'mathematics', 'biology', 'biochemistry',
        'molecular biology', 'cellular', 'genetics (non-medical)', 'ecology',
        'evolution', 'botany', 'zoology', 'marine', 'oceanography',
        'atmospheric', 'geology', 'geoscience', 'astronomy', 'astrophysics',
        'biophysics', 'organic chemistry', 'inorganic chemistry',
        'physical and theoretical chemistry', 'spectroscopy', 'catalysis',
        'colloid', 'surface chemistry', 'analytical chemistry',
        'nature and landscape', 'environmental science', 'earth',
        'planetary', 'agricultural', 'food science', 'nutrition',
        'forestry', 'aquatic', 'microbiology (non-medical)'
    ]
    if any(t in field_lower for t in natural_terms):
        return 'Natural Sciences'

    return 'Other'


if asjc_col:
    df['domain'] = df[asjc_col].apply(map_to_domain)
else:
    df['domain'] = 'Other'

print("\nDomain distribution across all institutions:")
print(df['domain'].value_counts())

print("\nDomain × institution breakdown:")
print(df.groupby(['institution', 'domain']).size().unstack(fill_value=0).to_string())

## 3. Feature Engineering Helpers

Same pipeline as nb43.

In [ ]:
def preprocess_text(text):
    if pd.isna(text):
        return ""
    return str(text).lower()


def build_venue_features(subset_df):
    vf = pd.DataFrame(index=subset_df.index)
    col_map = {
        'snip':                 'SNIP (publication year)',
        'snip_percentile':      'SNIP percentile (publication year) *',
        'citescore':            'CiteScore (publication year)',
        'citescore_percentile': 'CiteScore percentile (publication year) *',
        'sjr':                  'SJR (publication year)',
        'sjr_percentile':       'SJR percentile (publication year) *',
    }
    for feat, col in col_map.items():
        vf[feat] = pd.to_numeric(subset_df[col], errors='coerce') if col in subset_df.columns else np.nan

    pct_cols = ['snip_percentile', 'citescore_percentile', 'sjr_percentile']
    vf['avg_venue_percentile'] = vf[pct_cols].mean(axis=1)
    vf['is_top_journal'] = (
        (vf['snip_percentile'] >= 90) |
        (vf['citescore_percentile'] >= 90) |
        (vf['sjr_percentile'] >= 90)
    ).astype(int)
    vf['venue_score_composite'] = (
        vf['snip'].fillna(0) * 0.33 +
        vf['citescore'].fillna(0) * 0.33 +
        vf['sjr'].fillna(0) * 0.34
    )
    for col in vf.columns:
        median_val = vf[col].median()
        vf[col] = vf[col].fillna(0 if pd.isna(median_val) else median_val)
    return vf


def build_author_features(subset_df):
    af = pd.DataFrame(index=subset_df.index)
    for feat, col in [
        ('num_authors',      'Number of Authors'),
        ('num_institutions', 'Number of Institutions'),
        ('num_countries',    'Number of Countries/Regions'),
    ]:
        af[feat] = pd.to_numeric(
            subset_df.get(col, pd.Series(np.nan, index=subset_df.index)), errors='coerce'
        )
    af['is_single_author']        = (af['num_authors'] == 1).astype(int)
    af['is_international_collab'] = (af['num_countries'] > 1).astype(int)
    af['is_multi_institution']    = (af['num_institutions'] > 1).astype(int)
    af['authors_per_institution'] = af['num_authors'] / af['num_institutions'].replace(0, 1)
    af['team_size_small']  = (af['num_authors'] <= 3).astype(int)
    af['team_size_medium'] = ((af['num_authors'] > 3) & (af['num_authors'] <= 10)).astype(int)
    af['team_size_large']  = (af['num_authors'] > 10).astype(int)
    for col in af.columns:
        median_val = af[col].median()
        af[col] = af[col].fillna(0 if pd.isna(median_val) else median_val)
    return af


def build_metadata_features(subset_df, pub_type_cols=None, source_type_cols=None):
    mf = pd.DataFrame(index=subset_df.index)
    mf['is_open_access'] = subset_df.get(
        'Open Access', pd.Series(np.nan, index=subset_df.index)
    ).notna().astype(int)
    mf['topic_prominence'] = pd.to_numeric(
        subset_df.get('Topic Prominence Percentile', pd.Series(np.nan, index=subset_df.index)),
        errors='coerce'
    ).fillna(0)
    pub_dummies = pd.get_dummies(
        subset_df.get('Publication type', pd.Series(dtype=str)), prefix='pubtype', dummy_na=False
    )
    if pub_type_cols is not None:
        pub_dummies = pub_dummies.reindex(columns=pub_type_cols, fill_value=0)
    src_dummies = pd.get_dummies(
        subset_df.get('Source type', pd.Series(dtype=str)), prefix='sourcetype', dummy_na=False
    )
    if source_type_cols is not None:
        src_dummies = src_dummies.reindex(columns=source_type_cols, fill_value=0)
    mf = pd.concat([mf, pub_dummies, src_dummies], axis=1)
    return mf, list(pub_dummies.columns), list(src_dummies.columns)


def build_feature_matrix(df_train, df_test_list, tfidf_kwargs=None):
    """Build full feature matrices for train + list of test sets.
    Returns (X_train, [X_test, ...], tfidf_vectorizer)
    """
    if tfidf_kwargs is None:
        tfidf_kwargs = dict(max_features=5000, ngram_range=(1, 2), min_df=5, max_df=0.8, stop_words='english')

    # TF-IDF
    tfidf = TfidfVectorizer(**tfidf_kwargs)
    tfidf_tr = tfidf.fit_transform(df_train['Abstract'].apply(preprocess_text))
    feat_names = [f'tfidf_{f}' for f in tfidf.get_feature_names_out()]
    text_tr = pd.DataFrame(tfidf_tr.toarray(), columns=feat_names, index=df_train.index)

    venue_tr = build_venue_features(df_train)
    author_tr = build_author_features(df_train)
    meta_tr, pub_cols, src_cols = build_metadata_features(df_train)

    X_tr = pd.concat([text_tr, venue_tr, author_tr, meta_tr], axis=1)

    X_tests = []
    for df_te in df_test_list:
        tfidf_te = tfidf.transform(df_te['Abstract'].apply(preprocess_text))
        text_te = pd.DataFrame(tfidf_te.toarray(), columns=feat_names, index=df_te.index)
        venue_te = build_venue_features(df_te)
        author_te = build_author_features(df_te)
        meta_te, _, _ = build_metadata_features(df_te, pub_cols, src_cols)
        X_te = pd.concat([text_te, venue_te, author_te, meta_te], axis=1)
        X_tests.append(X_te)

    return X_tr, X_tests, tfidf


def find_optimal_threshold(y_true, y_proba, thresholds=None):
    if thresholds is None:
        thresholds = np.arange(0.10, 0.90, 0.01)
    f1s = [f1_score(y_true, (y_proba >= t).astype(int)) for t in thresholds]
    best_idx = int(np.argmax(f1s))
    return thresholds[best_idx], f1s[best_idx]


def compute_metrics(y_true, y_proba, threshold):
    y_pred = (y_proba >= threshold).astype(int)
    return {
        'f1':        f1_score(y_true, y_pred),
        'roc_auc':   roc_auc_score(y_true, y_proba),
        'recall':    recall_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'threshold': threshold,
    }


def train_domain_models(X_train, y_train, domains_train, min_test_size=50):
    """Train one LogisticRegression per domain. Returns {domain: model}."""
    models = {}
    for domain in sorted(domains_train.unique()):
        mask = domains_train == domain
        X_d, y_d = X_train[mask], y_train[mask]
        if y_d.sum() < 10 or (len(y_d) - y_d.sum()) < 10:
            print(f"  Skipping {domain} (insufficient class balance in train)")
            continue
        m = LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1, class_weight='balanced')
        m.fit(X_d, y_d)
        models[domain] = m
        print(f"  Trained {domain:30s} — {len(X_d):,} train samples")
    return models


def selective_domain_segmentation(domain_models, baseline_model, X_test, y_test,
                                   domains_test, min_test_size=50):
    """
    For each domain, compare optimised domain model F1 vs baseline F1.
    Use the domain model only where it wins; otherwise keep baseline.
    Returns (y_pred_final, y_proba_final, per_domain_log).
    """
    y_proba_base = baseline_model.predict_proba(X_test)[:, 1]
    opt_t_base, _ = find_optimal_threshold(y_test, y_proba_base)
    y_pred_final  = (y_proba_base >= opt_t_base).astype(float)
    y_proba_final = y_proba_base.copy()

    log = []
    for domain, model in domain_models.items():
        mask = domains_test == domain
        if mask.sum() < min_test_size:
            continue
        idx    = np.where(mask.values)[0]
        X_d    = X_test[mask]
        y_d    = y_test[mask]

        # Domain model
        y_proba_d = model.predict_proba(X_d)[:, 1]
        opt_t_d, f1_d = find_optimal_threshold(y_d, y_proba_d)

        # Baseline on this domain
        y_proba_b_d = y_proba_base[idx]
        opt_t_b_d, f1_b_d = find_optimal_threshold(y_d, y_proba_b_d)

        if f1_d > f1_b_d:
            y_pred_final[idx]  = (y_proba_d >= opt_t_d).astype(float)
            y_proba_final[idx] = y_proba_d
            action = 'DOMAIN'
        else:
            action = 'BASELINE'

        log.append({
            'domain':      domain,
            'n_test':      mask.sum(),
            'baseline_f1': f1_b_d,
            'domain_f1':   f1_d,
            'delta':       f1_d - f1_b_d,
            'used':        action,
        })

    return y_pred_final, y_proba_final, pd.DataFrame(log)

print("Helpers defined.")

## 4. Scenario A — 2015-2017 Merged Train

Same training window as nb43 for a direct apples-to-apples comparison.

In [ ]:
print("=" * 70)
print("SCENARIO A — Train: all-unis 2015-2017 | Test: AUB 2018-2020")
print("=" * 70)

TRAIN_YEARS_A = [2015, 2016, 2017]
TEST_YEARS    = [2018, 2019, 2020]

df_train_a  = df[df['Year'].isin(TRAIN_YEARS_A)].copy()
df_test_all = df[df['Year'].isin(TEST_YEARS)].copy()
df_test_aub = df_test_all[df_test_all['institution'] == 'AUB'].copy()

print(f"Train: {len(df_train_a):,} papers")
print(df_train_a['institution'].value_counts().to_string())
print(f"\nTest (AUB-only): {len(df_test_aub):,} papers")
print(f"Test (all-unis): {len(df_test_all):,} papers")

print("\nDomain distribution in train set (Scenario A):")
print(df_train_a['domain'].value_counts())

In [ ]:
# Build features
X_train_a, [X_test_aub_a, X_test_all_a], tfidf_a = build_feature_matrix(
    df_train_a, [df_test_aub, df_test_all]
)

# Build targets (threshold from training citations)
cit_thresh_a = df_train_a['Citations'].quantile(0.75)
y_train_a    = (df_train_a['Citations'] >= cit_thresh_a).astype(int)
y_test_aub_a = (df_test_aub['Citations'] >= cit_thresh_a).astype(int)
y_test_all_a = (df_test_all['Citations'] >= cit_thresh_a).astype(int)

domains_train_a   = df_train_a['domain']
domains_test_aub_a = df_test_aub['domain']
domains_test_all_a = df_test_all['domain']

print(f"Citation threshold (75th pct of train): {cit_thresh_a:.0f}")
print(f"Feature matrix: {X_train_a.shape}")
print(f"Train high-impact: {y_train_a.mean()*100:.1f}%")
print(f"Test (AUB) high-impact: {y_test_aub_a.mean()*100:.1f}%")

In [ ]:
print("Training baseline (universal) model...")
baseline_a = LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1, class_weight='balanced')
baseline_a.fit(X_train_a, y_train_a)

proba_base_aub_a = baseline_a.predict_proba(X_test_aub_a)[:, 1]
opt_t, opt_f1 = find_optimal_threshold(y_test_aub_a, proba_base_aub_a)
metrics_base_aub_a = compute_metrics(y_test_aub_a, proba_base_aub_a, opt_t)

print(f"\nBaseline (merged train, AUB test):")
print(f"  F1={metrics_base_aub_a['f1']*100:.2f}%  ROC-AUC={metrics_base_aub_a['roc_auc']*100:.2f}%  "
      f"Recall={metrics_base_aub_a['recall']*100:.2f}%  Precision={metrics_base_aub_a['precision']*100:.2f}%")

In [ ]:
print("Training domain-specific models (Scenario A)...")
domain_models_a = train_domain_models(X_train_a, y_train_a, domains_train_a)

In [ ]:
print("Running selective domain segmentation on AUB test set...\n")
y_pred_sel_a, y_proba_sel_a, log_a = selective_domain_segmentation(
    domain_models_a, baseline_a, X_test_aub_a, y_test_aub_a, domains_test_aub_a
)

print("Per-domain decision:")
print(log_a.to_string(index=False))

# Overall selective metrics
opt_t_sel_a, _ = find_optimal_threshold(y_test_aub_a, y_proba_sel_a)
y_pred_final_a = (y_proba_sel_a >= opt_t_sel_a).astype(int)
metrics_sel_aub_a = compute_metrics(y_test_aub_a, y_proba_sel_a, opt_t_sel_a)

print(f"\nSelective domain segmentation (AUB test):")
print(f"  F1={metrics_sel_aub_a['f1']*100:.2f}%  ROC-AUC={metrics_sel_aub_a['roc_auc']*100:.2f}%  "
      f"Recall={metrics_sel_aub_a['recall']*100:.2f}%  Precision={metrics_sel_aub_a['precision']*100:.2f}%")

In [ ]:
# Also evaluate on all-unis test set
print("Running selective domain segmentation on all-unis test set...\n")
y_pred_sel_all_a, y_proba_sel_all_a, log_all_a = selective_domain_segmentation(
    domain_models_a, baseline_a, X_test_all_a, y_test_all_a, domains_test_all_a
)

opt_t_sel_all_a, _ = find_optimal_threshold(y_test_all_a, y_proba_sel_all_a)
metrics_sel_all_a = compute_metrics(y_test_all_a, y_proba_sel_all_a, opt_t_sel_all_a)

print(f"Selective domain segmentation (all-unis test):")
print(f"  F1={metrics_sel_all_a['f1']*100:.2f}%  ROC-AUC={metrics_sel_all_a['roc_auc']*100:.2f}%  "
      f"Recall={metrics_sel_all_a['recall']*100:.2f}%  Precision={metrics_sel_all_a['precision']*100:.2f}%")

## 5. Scenario B — 2010-2017 Merged Train

Maximum training data per domain; same train window used in nb42 but now with all universities.

In [ ]:
print("=" * 70)
print("SCENARIO B — Train: all-unis 2010-2017 | Test: AUB 2018-2020")
print("=" * 70)

TRAIN_YEARS_B = list(range(2010, 2018))

df_train_b = df[df['Year'].isin(TRAIN_YEARS_B)].copy()

print(f"Train: {len(df_train_b):,} papers")
print(df_train_b['institution'].value_counts().to_string())
print(f"\nDomain distribution in train set (Scenario B):")
print(df_train_b['domain'].value_counts())
print(f"\nTrain increase over Scenario A: +{len(df_train_b)-len(df_train_a):,} papers")

In [ ]:
X_train_b, [X_test_aub_b, X_test_all_b], tfidf_b = build_feature_matrix(
    df_train_b, [df_test_aub, df_test_all]
)

cit_thresh_b  = df_train_b['Citations'].quantile(0.75)
y_train_b     = (df_train_b['Citations'] >= cit_thresh_b).astype(int)
y_test_aub_b  = (df_test_aub['Citations'] >= cit_thresh_b).astype(int)
y_test_all_b  = (df_test_all['Citations'] >= cit_thresh_b).astype(int)

domains_train_b    = df_train_b['domain']
domains_test_aub_b = df_test_aub['domain']
domains_test_all_b = df_test_all['domain']

print(f"Citation threshold (75th pct of train): {cit_thresh_b:.0f}")
print(f"Feature matrix: {X_train_b.shape}")

In [ ]:
print("Training baseline (universal) model — Scenario B...")
baseline_b = LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1, class_weight='balanced')
baseline_b.fit(X_train_b, y_train_b)

proba_base_aub_b = baseline_b.predict_proba(X_test_aub_b)[:, 1]
opt_t_b, _ = find_optimal_threshold(y_test_aub_b, proba_base_aub_b)
metrics_base_aub_b = compute_metrics(y_test_aub_b, proba_base_aub_b, opt_t_b)

print(f"\nBaseline (Scenario B, AUB test):")
print(f"  F1={metrics_base_aub_b['f1']*100:.2f}%  ROC-AUC={metrics_base_aub_b['roc_auc']*100:.2f}%")

In [ ]:
print("Training domain-specific models (Scenario B)...")
domain_models_b = train_domain_models(X_train_b, y_train_b, domains_train_b)

In [ ]:
print("Running selective domain segmentation on AUB test set (Scenario B)...\n")
y_pred_sel_b, y_proba_sel_b, log_b = selective_domain_segmentation(
    domain_models_b, baseline_b, X_test_aub_b, y_test_aub_b, domains_test_aub_b
)

print("Per-domain decision:")
print(log_b.to_string(index=False))

opt_t_sel_b, _ = find_optimal_threshold(y_test_aub_b, y_proba_sel_b)
metrics_sel_aub_b = compute_metrics(y_test_aub_b, y_proba_sel_b, opt_t_sel_b)

print(f"\nSelective domain segmentation (Scenario B, AUB test):")
print(f"  F1={metrics_sel_aub_b['f1']*100:.2f}%  ROC-AUC={metrics_sel_aub_b['roc_auc']*100:.2f}%  "
      f"Recall={metrics_sel_aub_b['recall']*100:.2f}%  Precision={metrics_sel_aub_b['precision']*100:.2f}%")

In [ ]:
# All-unis test for Scenario B
y_pred_sel_all_b, y_proba_sel_all_b, log_all_b = selective_domain_segmentation(
    domain_models_b, baseline_b, X_test_all_b, y_test_all_b, domains_test_all_b
)

opt_t_sel_all_b, _ = find_optimal_threshold(y_test_all_b, y_proba_sel_all_b)
metrics_sel_all_b = compute_metrics(y_test_all_b, y_proba_sel_all_b, opt_t_sel_all_b)

print(f"Selective domain segmentation (Scenario B, all-unis test):")
print(f"  F1={metrics_sel_all_b['f1']*100:.2f}%  ROC-AUC={metrics_sel_all_b['roc_auc']*100:.2f}%")

## 6. Summary Table

In [ ]:
AUB_BASELINE_F1  = 0.6255
NB43_MERGED_F1   = 0.6253
NB42_DOMSEG_F1   = 0.6333

rows = [
    {'Model':                    'AUB-only baseline (ref)',
     'Train':                    'AUB 2015-2017',
     'Test':                     'AUB 2018-2020',
     'F1':                       AUB_BASELINE_F1,
     'vs AUB baseline':          0.0},
    {'Model':                    'Merged, no domain seg (nb43)',
     'Train':                    'All-unis 2015-2017',
     'Test':                     'AUB 2018-2020',
     'F1':                       NB43_MERGED_F1,
     'vs AUB baseline':          NB43_MERGED_F1 - AUB_BASELINE_F1},
    {'Model':                    'Domain seg, AUB-only (nb42)',
     'Train':                    'AUB 2010-2017',
     'Test':                     'AUB 2018-2020',
     'F1':                       NB42_DOMSEG_F1,
     'vs AUB baseline':          NB42_DOMSEG_F1 - AUB_BASELINE_F1},
    {'Model':                    'Merged + domain seg — Scen A (this nb)',
     'Train':                    'All-unis 2015-2017',
     'Test':                     'AUB 2018-2020',
     'F1':                       metrics_sel_aub_a['f1'],
     'vs AUB baseline':          metrics_sel_aub_a['f1'] - AUB_BASELINE_F1},
    {'Model':                    'Merged + domain seg — Scen B (this nb)',
     'Train':                    'All-unis 2010-2017',
     'Test':                     'AUB 2018-2020',
     'F1':                       metrics_sel_aub_b['f1'],
     'vs AUB baseline':          metrics_sel_aub_b['f1'] - AUB_BASELINE_F1},
]

summary = pd.DataFrame(rows)

print("=" * 80)
print("FULL COMPARISON — AUB Test Set (2018-2020)")
print("=" * 80)
print(f"\n{'Model':<45} {'Train':<22} {'F1':>8} {'Δ vs baseline':>15}")
print("-" * 95)
for _, r in summary.iterrows():
    delta = r['vs AUB baseline']
    best_marker = " ◄ BEST" if r['F1'] == summary['F1'].max() else ""
    print(f"  {r['Model']:<43} {r['Train']:<22} {r['F1']*100:>7.2f}%  {delta*100:>+9.2f}pp{best_marker}")
print("=" * 80)

best_row = summary.loc[summary['F1'].idxmax()]
print(f"\nBest model: {best_row['Model']}")
print(f"F1: {best_row['F1']*100:.2f}%  (+{best_row['vs AUB baseline']*100:.2f}pp vs AUB baseline)")

In [ ]:
# Visualise
fig, ax = plt.subplots(figsize=(12, 6))

colors = ['#4e79a7', '#f28e2b', '#59a14f', '#e15759', '#76b7b2']
bars = ax.barh(summary['Model'], summary['F1'] * 100, color=colors, alpha=0.85, edgecolor='white')

# Add value labels
for bar, row in zip(bars, summary.itertuples()):
    delta = row._4  # 'vs AUB baseline'
    sign  = '+' if delta >= 0 else ''
    ax.text(
        bar.get_width() + 0.1, bar.get_y() + bar.get_height() / 2,
        f"{bar.get_width():.2f}%  ({sign}{delta*100:.2f}pp)",
        va='center', fontsize=10
    )

ax.axvline(AUB_BASELINE_F1 * 100, color='red', linestyle='--', linewidth=1.5,
           label=f'AUB baseline ({AUB_BASELINE_F1*100:.2f}%)')
ax.set_xlabel('F1 Score (%)', fontsize=12)
ax.set_title('Merged Data + Domain Segmentation — AUB Test Set', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.set_xlim(60, 68)
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
fig_dir = Path('../reports/figures')
fig_dir.mkdir(parents=True, exist_ok=True)
plt.savefig(fig_dir / 'merged_domain_segmentation_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Per-Domain Breakdown

In [ ]:
print("=" * 70)
print("PER-DOMAIN DECISIONS — Scenario A (all-unis 2015-2017 train)")
print("=" * 70)
if not log_a.empty:
    log_a_disp = log_a.copy()
    for col in ['baseline_f1', 'domain_f1', 'delta']:
        log_a_disp[col] = log_a_disp[col].apply(lambda x: f"{x*100:+.2f}pp" if col == 'delta' else f"{x*100:.2f}%")
    print(log_a_disp.to_string(index=False))
else:
    print("No domains met the minimum test-size threshold.")

print("\n" + "=" * 70)
print("PER-DOMAIN DECISIONS — Scenario B (all-unis 2010-2017 train)")
print("=" * 70)
if not log_b.empty:
    log_b_disp = log_b.copy()
    for col in ['baseline_f1', 'domain_f1', 'delta']:
        log_b_disp[col] = log_b_disp[col].apply(lambda x: f"{x*100:+.2f}pp" if col == 'delta' else f"{x*100:.2f}%")
    print(log_b_disp.to_string(index=False))
else:
    print("No domains met the minimum test-size threshold.")

## 8. Save Artifacts

In [ ]:
models_dir  = Path('../models')
metrics_dir = Path('../reports/metrics')
models_dir.mkdir(exist_ok=True)
metrics_dir.mkdir(parents=True, exist_ok=True)

# Save models
with open(models_dir / 'merged_domain_seg_baseline_scen_a.pkl', 'wb') as f:
    pickle.dump(baseline_a, f)
with open(models_dir / 'merged_domain_seg_domain_models_scen_a.pkl', 'wb') as f:
    pickle.dump(domain_models_a, f)
with open(models_dir / 'merged_domain_seg_baseline_scen_b.pkl', 'wb') as f:
    pickle.dump(baseline_b, f)
with open(models_dir / 'merged_domain_seg_domain_models_scen_b.pkl', 'wb') as f:
    pickle.dump(domain_models_b, f)

# Save summary CSV
summary.to_csv(metrics_dir / 'merged_domain_segmentation_summary.csv', index=False)
if not log_a.empty:
    log_a.to_csv(metrics_dir / 'merged_domain_seg_per_domain_scen_a.csv', index=False)
if not log_b.empty:
    log_b.to_csv(metrics_dir / 'merged_domain_seg_per_domain_scen_b.csv', index=False)

print("Saved:")
print("  models/merged_domain_seg_baseline_scen_a.pkl")
print("  models/merged_domain_seg_domain_models_scen_a.pkl")
print("  models/merged_domain_seg_baseline_scen_b.pkl")
print("  models/merged_domain_seg_domain_models_scen_b.pkl")
print("  reports/metrics/merged_domain_segmentation_summary.csv")
print("  reports/figures/merged_domain_segmentation_comparison.png")